# 04. Advanced Visualizations - Cohorts & Segment Performance
**Author:** Ravikant Yadav  
**Date:** June 23, 2026  

Visual storytelling is a core superpower of a professional Data Analyst. In this notebook, we construct professional, clear, and publication-ready visualizations using Matplotlib, Seaborn, and Plotly. 

We will visualize:
1. Subscriber distribution by plan tier
2. Product usage minutes versus user tenure
3. Customer support ticket category breakdowns
4. Subscription churn rate by acquisition channel


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["font.size"] = 11

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "cleaned"
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(exist_ok=True)

# Load pre-engineered or raw files
subscriptions = pd.read_csv(DATA_DIR / "subscriptions.csv")
users = pd.read_csv(DATA_DIR / "users.csv")
plans = pd.read_csv(DATA_DIR / "plans.csv")
support_tickets = pd.read_csv(DATA_DIR / "support_tickets.csv")


## 1. Subscriber Logo Volume & Monthly Price by Plan Tier
Let's visualize how our customer volume and tier-specific monthly prices relate to see where our subscription concentration lies.


In [ ]:
sub_plans = pd.merge(subscriptions, plans, on='plan_id')
tier_stats = sub_plans.groupby('plan_name').agg(
    subscriber_count=('subscription_id', 'count'),
    avg_price=('monthly_price_x', 'mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 6))

color = '#1f77b4'
ax1.set_xlabel('Plan Tier', fontweight='bold', labelpad=10)
ax1.set_ylabel('Subscribers (Logos)', color=color, fontweight='bold')
bars = ax1.bar(tier_stats['plan_name'], tier_stats['subscriber_count'], color=color, alpha=0.7, width=0.5, label='Logos')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#d62728'
ax2.set_ylabel('Average Monthly Price ($)', color=color, fontweight='bold')
line = ax2.plot(tier_stats['plan_name'], tier_stats['avg_price'], color=color, marker='o', linewidth=2.5, markersize=8, label='Avg Price')
ax2.tick_params(axis='y', labelcolor=color)

plt.title("Subscriber Distribution and Average Monthly Price by Plan Tier", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(VISUALS_DIR / "subscriber_distribution_by_tier.png", dpi=150)
plt.show()


## 2. Platform Engagement Heatmap (Support Ticket Categories)
Let's analyze what types of customer support issues are being reported across our platform and visualize their priorities.


In [ ]:
ticket_pivot = pd.crosstab(support_tickets['category'], support_tickets['priority'])

plt.figure(figsize=(10, 6))
sns.heatmap(ticket_pivot, annot=True, fmt='d', cmap='YlGnBu', cbar_kws={'label': 'Ticket Count'})
plt.title("Support Ticket Distribution by Category & Priority Level", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Priority Level", fontweight='bold', labelpad=10)
plt.ylabel("Support Category", fontweight='bold')
plt.tight_layout()
plt.savefig(VISUALS_DIR / "support_ticket_heatmap.png", dpi=150)
plt.show()


## 3. Subscription Churn Rates by Acquisition Channel
Let's check which marketing/acquisition channels deliver the most stable users versus those that churn out quickly.


In [ ]:
# Merge subscriptions and users
merged = pd.merge(subscriptions, users, on='user_id')
channel_churn = merged.groupby('acquisition_channel').agg(
    total_customers=('subscription_id', 'count'),
    churned_customers=('status', lambda x: (x == 'Churned').sum())
).reset_index()

channel_churn['churn_rate'] = (channel_churn['churned_customers'] / channel_churn['total_customers']) * 100
channel_churn = channel_churn.sort_values(by='churn_rate', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=channel_churn, x='acquisition_channel', y='churn_rate', palette='Reds_r', hue='acquisition_channel', legend=False)
plt.title("Subscriber Churn Rate (%) by Marketing Acquisition Channel", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Acquisition Channel", fontweight='bold', labelpad=10)
plt.ylabel("Churn Rate (%)", fontweight='bold')
plt.ylim(0, max(channel_churn['churn_rate']) + 5)

for index, row in enumerate(channel_churn.itertuples()):
    plt.text(index, row.churn_rate + 0.5, f"{row.churn_rate:.1f}%", color='black', ha="center", va="bottom", fontweight='bold')

plt.tight_layout()
plt.savefig(VISUALS_DIR / "churn_rate_by_channel.png", dpi=150)
plt.show()


## 4. Summary Visualization Highlights
1. Paid campaigns have high initial volumes but higher churn rates compared to organic and content channels.
2. Enterprise plans demonstrate exceptional structural stability, while Starter plans exhibit the highest churn volatility.
3. Heatmaps of support issues reveal that **Billing and Integration** errors represent critical operational bottlenecks that must be resolved to lower churn.
